#**Taller 8 de Julio**

##**Importar Librerías**

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

##*Importar el Dataset*

In [ ]:
df = pd.read_csv(r'C:\Users\juan1\Downloads\Taller 8 julio\test_Y3wMUE5_7gLdaTN.csv')
df.head()

##**Identificación de Variable Objetivo**

Si estamos hablando de créditos, la variable objetivo sería el monto a prestar (LoanAmount)

In [ ]:
df.info()

df.isnull().sum()

df["LoanAmount"].value_counts()

## 1. Es un problema de clasificación o de regresión?

RTA: De regresión ya que la variable monto credito es una variable cuantitativa 

Contexto del problema: Vamos a evaluar 10 variables para determinar la viabilidad del monto que podemos prestarle a nuestros clientes. Nuestra variable objetivo es LoanAmount la cual está dada en miles de dolares. 

## 2. Exploración de datos EDA

##Revisión de valores nulos 

In [ ]:
# Crear una lista con las columnas específicas donde quieres eliminar los nulos
columnas_a_limpiar = ['Gender', 'Dependents', 'Self_Employed', 'LoanAmount', 'Loan_Amount_Term']

# Mostrar cuántas filas había antes de eliminar
filas_antes = df.shape[0]

# Eliminar las filas donde cualquiera de las columnas anteriores tenga un NaN
df = df.dropna(subset=columnas_a_limpiar)

# Mostrar el resultado del proceso
filas_despues = df.shape[0]
filas_eliminadas = filas_antes - filas_despues

print(f"Filas originales: {filas_antes}")
print(f"Filas eliminadas debido a valores NaN: {filas_eliminadas}")
print(f"Filas restantes en el dataset: {filas_despues}")

# Verificar que ya no queden nulos en esas columnas concretas
print("\nConteo de nulos actual en esas columnas:")
print(df[columnas_a_limpiar].isnull().sum())

In [ ]:
df.info()


In [ ]:
# Rellenar los valores NaN de la columna Credit_History con 0
df['Credit_History'] = df['Credit_History'].fillna(0)

# Verificar que ya no queden nulos en esta columna
print(f"Valores nulos restantes en 'Credit_History': {df['Credit_History'].isnull().sum()}")

##**Gráficos** 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar el estilo estético de los gráficos
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ==========================================================
# GRÁFICO 1: Correlación (Dispersión) entre LoanAmount y ApplicantIncome
# ==========================================================
sns.scatterplot(
    ax=axes[0],
    data=df,
    x='ApplicantIncome',
    y='LoanAmount',
    alpha=0.7,
    color='teal'
)
# Añadir una línea de tendencia para ver mejor la correlación
sns.regplot(
    ax=axes[0],
    data=df,
    x='ApplicantIncome',
    y='LoanAmount',
    scatter=False,
    color='red'
)
axes[0].set_title('Correlación: Ingreso del Solicitante vs Monto del Préstamo', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Ingreso del Solicitante (ApplicantIncome)')
axes[0].set_ylabel('Monto del Préstamo (LoanAmount)')

# ==========================================================
# GRÁFICO 2: Distribución de Educación vs Promedio de ApplicantIncome
# ==========================================================
# Calculamos el promedio de ingresos agrupado por nivel educativo
df_edu_income = df.groupby('Education')['ApplicantIncome'].mean().reset_index()

sns.barplot(
    ax=axes[1],
    data=df_edu_income,
    x='Education',
    y='ApplicantIncome',
    palette='Blues_d',
    hue='Education',
    legend=False
)
axes[1].set_title('Promedio de Ingresos según el Nivel Educativo', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nivel Educativo (Education)')
axes[1].set_ylabel('Promedio de ApplicantIncome')

# Ajustar el diseño para que no se traslapen los textos
plt.tight_layout()
plt.show()

## 3. Prepracion de datos

In [ ]:


# ==========================================================
# PASO 1: SEPARAR VARIABLES PREDICTORAS (X) Y OBJETIVO (y)
# ==========================================================
# X contiene todas las columnas excepto nuestra variable objetivo
X = df_encoded.drop(columns=['LoanAmount'])

# y contiene únicamente la variable objetivo
y = df_encoded['LoanAmount']

# ==========================================================
# PASO 2: DIVIDIR EN ENTRENAMIENTO Y PRUEBA (Train/Test Split)
# ==========================================================
# Es una buena práctica dividir antes de escalar para evitar la "fuga de datos" (data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================================
# PASO 3: ESCALAMIENTO DE LOS DATOS (StandardScaler)
# ==========================================================
scaler = StandardScaler()

# Ajustar el escalador y transformar los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Transformar los datos de prueba usando el MISMO escalador
X_test_scaled = scaler.transform(X_test)

# Convertir de vuelta a DataFrame solo para verificar visualmente cómo quedaron
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)

print("--- Datos de entrenamiento ESCALADOS (Primeras filas) ---")
print(X_train_scaled_df.head())



## 4. Preparación de los datos para el modelo

In [ ]:


# ==========================================================
# PASO 1: ELIMINAR LOAN_ID Y SEPARAR PREDICTORAS (X) Y OBJETIVO (y)
# ==========================================================
# Eliminamos tanto 'LoanAmount' (variable objetivo) como 'Loan_ID' (identificador)
X = df_encoded.drop(columns=['LoanAmount', 'Loan_ID'], errors='ignore')

# y contiene únicamente la variable objetivo
y = df_encoded['LoanAmount']

# ==========================================================
# PASO 2: DIVIDIR EN ENTRENAMIENTO Y PRUEBA (Train/Test Split)
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================================
# PASO 3: ESCALAMIENTO DE LOS DATOS (StandardScaler)
# ==========================================================
scaler = StandardScaler()

# Ajustar el escalador y transformar los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Transformar los datos de prueba usando el MISMO escalador
X_test_scaled = scaler.transform(X_test)

# Convertir de vuelta a DataFrame solo para verificar qué columnas quedaron
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)

print("--- Variables predictoras finales que entrarán al modelo ---")
print(list(X.columns))

print("\n--- Vista previa de los datos escalados (sin Loan_ID) ---")
print(X_train_scaled_df.head(2))

--- Variables predictoras finales que entrarán al modelo ---
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History', 'Property_Area_Semiurban', 'Property_Area_Urban']

--- Vista previa de los datos escalados (sin Loan_ID) ---
     Gender   Married  Dependents  Education  Self_Employed  ApplicantIncome  \
0  0.511182 -1.325987   -0.772654   0.517375      -0.339935        -0.270461   
1 -1.956252 -1.325987   -0.772654   0.517375      -0.339935        -0.574448   

   CoapplicantIncome  Loan_Amount_Term  Credit_History  \
0          -0.610410          0.260074        0.523557   
1           0.411725          0.260074       -1.910013   

   Property_Area_Semiurban  Property_Area_Urban  
0                -0.652791            -0.807041  
1                -0.652791             1.239094  


## 5 Evaluación del modelo 

In [20]:

# ==========================================================
# PASO 1: ENTRENAR EL MODELO DE REGRESIÓN LINEAL
# ==========================================================
# Creamos la instancia del modelo
modelo = LinearRegression()

# Entrenamos el modelo usando los datos ESCALADOS de entrenamiento
modelo.fit(X_train_scaled, y_train)

# ==========================================================
# PASO 2: REALIZAR PREDICCIONES
# ==========================================================
# Hacemos las predicciones usando los datos de prueba (test)
y_pred = modelo.predict(X_test_scaled)

# ==========================================================
# PASO 3: EVALUACIÓN CON MAE, MSE Y R CUADRADO
# ==========================================================
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # El RMSE es más fácil de interpretar que el MSE
r2 = r2_score(y_test, y_pred)

print("--- MÉTRICAS DE EVALUACIÓN DEL MODELO ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R²): {r2:.4f}")

--- MÉTRICAS DE EVALUACIÓN DEL MODELO ---
Mean Absolute Error (MAE): 38.00
Mean Squared Error (MSE): 2677.74
Root Mean Squared Error (RMSE): 51.75
R-squared (R²): 0.2374


1. ¿Qué tan bueno es el modelo?

Análisis de $R^2$: El Coeficiente de Determinación es de 0.2374. Esto significa que las variables independientes actuales (ingresos, educación, etc.) solo logran explicar el 23.74% de la variación en el monto del préstamo (LoanAmount). El 76.26% restante del comportamiento se debe a factores que el modelo lineal no está capturando.

Análisis del error (MAE y RMSE): El MAE nos indica que, en promedio, las predicciones del modelo fallan por 38 unidades (si el dataset está expresado en miles, serían $38,000). Además, la diferencia entre el MAE (38.00) y el RMSE (51.75) confirma que hay valores atípicos importantes (outliers) en los ingresos o en los montos solicitados, los cuales desestabilizan y penalizan la precisión del algoritmo.

2. ¿Qué variable parece influir más en la predicción?

Como aplicamos una estandarización de los datos (StandardScaler) antes de entrenar la Regresión Lineal, la importancia se mide por la magnitud de los coeficientes del modelo.



En este problema de riesgo crediticio, la variable que más influye es ApplicantIncome (el ingreso del solicitante) de forma positiva. Esto se debe a la lógica natural del negocio bancario: la capacidad de endeudamiento y el monto máximo que una entidad financiera está dispuesta a otorgar dependen directamente de los ingresos estables que demuestre tener la persona. A mayores ingresos, mayor es el valor del préstamo autorizado.

3. ¿Qué mejorarías si tuvieras más tiempo?

Para elevar ese $R^2$ drásticamente del 23% a un rendimiento óptimo, propondría las siguientes tres estrategias:


Implementar modelos no lineales basados en árboles: La Regresión Lineal asume que los datos siguen una línea recta perfecta, lo cual rara vez ocurre en finanzas. Utilizaría algoritmos como Random Forest Regressor o Gradient Boosting (XGBoost), que se adaptan mucho mejor a patrones curvos y a la interacción compleja entre variables categóricas y numéricas.


Ingeniería de características (Feature Engineering): Crearía variables combinadas más potentes para el negocio. Por ejemplo, una columna Total_Income que sume los ingresos del solicitante (ApplicantIncome) y del co-solicitante (CoapplicantIncome), ya que los bancos evalúan el músculo financiero del hogar completo y no solo de un individuo.


Tratamiento estricto de Outliers: Aplicaría transformaciones logarítmicas o técnicas de acotación (clipping) sobre las columnas de ingresos para suavizar los valores extremadamente altos. Esto evitaría que esos pocos clientes con ingresos gigantescos distorsionen la recta de regresión y degraden las métricas globales de evaluación.



